# M4 summary: the goals and the tables behind them

Milestone 4 set five goals for the model and kept one safety rule from Milestone 3. This notebook reads
the result files the Milestone 4 notebooks saved under `results/m4/` and turns them into the report's
tables. It trains nothing, scores nothing and opens no photo or clip: every number below is read from a
saved file, and every table names its source. It runs on a laptop's CPU in seconds and writes the tables
to `results/m4/summary/`.

| Goal | Pass mark | Measured on |
|---|---|---|
| G1 Scenery | the gap between top-1 as is and top-1 with the scenery swapped, on the same photos, is at most half of M3's gap | validation photos with a usable mask |
| G2 Most classes | at least 80 of the measurable species reach a validation F1 of 0.5 | validation photos |
| G3 Existing tools | on B1's species, our top-1 and macro-F1 are each higher than B1's, or the 95% photographer-bootstrap interval of the difference includes 0 | the 2022 upload slice of validation |
| G4 False snakes | at most 5% of the non-snake test photos are called a snake, while at least 99% of validation snakes are still found | the non-snake test split, scored once; validation snakes |
| G5 Video | on real clips, the shown species changes at most once per clip after the first answer; time to the first answer and the share of right final answers are reported | the real test clips that answer; a clip that never answers is reported and does not pass, and with no answering clip G5 is not met |
| Safety | WHO recall's lower bound on calibration stays at 0.95 or above | calibration photos |

The pass marks are the design's. Where a result file holds a mark (G4, safety), the code reads it from
the file; the others are constants in the first cell. The non-snake test split was scored once, and its
result file is all this notebook reads of it.

In [1]:
import ast, json, os
from pathlib import Path
from xml.etree import ElementTree

import numpy as np
import pandas as pd
import yaml
from scipy.stats import binomtest

RESULTS = Path(os.environ.get("M4_RESULTS", "../results/m4"))
SUMMARY = Path(os.environ.get("M4_SUMMARY", RESULTS / "summary"))
NAMED_AS = "results/m4"  # how the tables name a source file
FULL_RUNS = ("A1", "H2", "A3")
SHIPPED = "A3"  # the model the pre-registered rule picked; section 2 must give it back
RUN_KEY, M3_KEY = "M4_FULL", "M3 final"  # the keys the full runs saved their rows under
G1_SHARE_OF_M3 = 0.5
G2_SPECIES, G2_F1 = 80, 0.5
G2_LEVEL = f"f1_at_least_{G2_F1}"
G3_SET = "upload_2022_slice"
G5_MAX_CHANGES = 1
NO_ANSWER = "no answer"  # a clip that never shows a species: reported, never within G5
M3_DEFINITION = "macro_f1_m3_definition"  # M3's whole-slice macro-F1, reported and not judged
PILOT_PASS_AT = 40  # notebook 22: a part passes with this many of its boxes right
BEFORE_GATE = "M3 box"  # Milestone 3's gate, scored on the same test photos for the record
LOOKALIKES = ("caecilian", "earthworm", "eel", "legless_lizard")
VIDEO_MODELS = ("M3", "A3")
SEA_SNAKES = "sea snakes and sea kraits"  # lookalike_recall.csv's group
SUMMARY.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)


def read_json(path):
    return json.loads((RESULTS / path).read_text())


def named(path):
    """A source file as the tables name it."""
    return f"{NAMED_AS}/{path}"


def save(table, name):
    table.to_csv(SUMMARY / f"{name}.csv", index=False)
    print(f"wrote {named(f'summary/{name}.csv')}: {len(table)} rows")
    return table


def interval_text(diff, low, high):
    return f"{diff:+.3f} ({low:+.3f} to {high:+.3f})"


print(f"reading {RESULTS}, writing {SUMMARY}")

reading ../results/m4, writing ../results/m4/summary


## 1. The background arms, on the subset

Notebook 18 trained the ways of seeing the snake (the arms) on the experiment subset with the Milestone 3
recipe, one seed each, and picked one with a fixed rule. An arm qualifies if its macro-F1 and its WHO
recall stay above floors set from A0, Milestone 3's final model; of A0 and the qualifying arms, the
smallest gap between top-1 as is and with the scenery swapped wins, and a tie goes to the higher
macro-F1. The cell applies the rule again and checks it gives `pick.json`'s winner. Top-1 here is on the
validation photos with a usable mask, the photos the scenery test uses.

In [2]:
ARM_RECIPES = {"A0": "Milestone 3's final model: the detector's box crop, not retrained",
               "A1": "box crop, snake pasted onto another species' scenery half the time in training",
               "A2": "snake only: everything outside the predicted mask filled grey",
               "A3": "crop to the predicted mask's box, scenery swapped half the time in training"}
arms = pd.read_csv(RESULTS / "arms/arms.csv")
pick = read_json("arms/pick.json")
qualifies = (arms["macro_f1"] >= pick["f1_floor"]) & (arms["who_recall_dr0"] >= pick["who_floor"])
assert (qualifies == arms["qualifies"]).all(), "arms.csv's qualifies column disagrees with the floors"
order = arms[qualifies].sort_values(["gap", "macro_f1"], ascending=[True, False])
assert order["arm"].iloc[0] == pick["winner"], "the rule does not give pick.json's winner"
for arm, folder in (("A0", "A0_check"), ("A1", "M4_A1"), ("A2", "M4_A2"), ("A3", "M4_A3")):
    row = read_json(f"arms/{folder}/arm_row.json")
    assert np.isclose(arms.set_index("arm").loc[arm, "gap"], row["gap"]), f"{arm}: arm_row.json differs"
arm_table = save(arms.assign(recipe=arms["arm"].map(ARM_RECIPES), f1_floor=pick["f1_floor"],
                             who_floor=pick["who_floor"], winner=arms["arm"].eq(pick["winner"]),
                             source=named("arms/arms.csv") + "; " + named("arms/pick.json"))
                 [["arm", "run", "recipe", "front_end", "macro_f1", "who_recall_dr0", "top1", "top1_swapped",
                   "top1_blanked", "gap", "swap_photos", "f1_floor", "who_floor", "qualifies", "winner",
                   "source"]], "arms")
print("rule:", pick["rule"])
print("winner:", pick["winner"], "| qualifying, by gap:", list(order["arm"]))
arm_table.drop(columns=["recipe", "source"]).set_index("arm")

wrote results/m4/summary/arms.csv: 4 rows
rule: an arm qualifies if its macro-F1 >= A0's - 0.005 and its WHO recall (DR0, refit on calibration, lower-bound fit) >= A0's - 0.006; of A0 and the qualifying arms the smallest gap between top-1 as is and with the scenery swapped wins (same photos); a tie goes to the higher macro-F1
winner: A3 | qualifying, by gap: ['A3', 'A1', 'A0']


,run,front_end,macro_f1,who_recall_dr0,top1,top1_swapped,top1_blanked,gap,swap_photos,f1_floor,who_floor,qualifies,winner
arm,,,,,,,,,,,,,
A0,H1_top1_s1,F1,0.367,0.948,0.628,0.460,0.128,0.168,15977,0.362,0.942,True,False
A1,M4_A1,F1,0.372,0.945,0.631,0.538,0.140,0.093,15977,0.362,0.942,True,False
A2,M4_A2,F3,0.333,0.959,0.595,0.589,0.015,0.006,15977,0.362,0.942,False,False
A3,M4_A3,F1m,0.371,0.945,0.640,0.577,0.129,0.063,15977,0.362,0.942,True,True


## 2. The three full runs and the shipping rule

Three recipes were trained on all the training photos: A1 (box crop, scenery swapped), H2 (A1 plus the
detail crop, a second and closer look the model picks itself) and A3 (the arm pick's winner, the crop to
the predicted mask's box). The model the app ships was chosen by a rule fixed before the H2 and A3 runs
finished: among the full runs whose calibration WHO lower bound meets the safety mark, the one passing
the most of G1, G2 and G3; on a tie the smaller swap gap, then the higher macro-F1. The cell applies the
rule to the files and stops if it does not give the model that was shipped.

G3's verdicts come from each run's `b1_62.json`. The cell reads each difference again with notebook 21's
rule (higher; tied when the interval includes 0; lower) and checks it agrees with the file.

In [3]:
M3_ROBUSTNESS = "arms/A0_check/robustness_metrics.json"
M3_DECISION = "arms/A0_check/decision_metrics.json"


def swap_gap(robustness):
    """G1's gap: top-1 as is minus top-1 with the scenery swapped, on the same photos."""
    swapped = robustness["conditions"]["swapped"]
    return swapped["as_is_same_photos"]["top1"] - swapped["top1"]


def standing(interval):
    """Notebook 21's reading of a difference with B1."""
    if interval["diff"] > 0:
        return "higher"
    return "tied" if interval["low"] <= 0 <= interval["high"] else "lower"


m3_gap = swap_gap(read_json(M3_ROBUSTNESS))
g1_mark = m3_gap * G1_SHARE_OF_M3


def full_run_row(label):
    """Every goal number of one full run, read from its folder."""
    folder = f"full/{label}"
    final, robust = read_json(f"{folder}/final_metrics.json"), read_json(f"{folder}/robustness_metrics.json")
    counts, b1 = read_json(f"{folder}/per_class_counts.json"), read_json(f"{folder}/b1_62.json")
    decision, manifest = read_json(f"{folder}/decision_metrics.json"), read_json(f"{folder}/manifest.json")
    config = yaml.safe_load((RESULTS / folder / "config.yaml").read_text())
    assert manifest["label"] == label and final["run"]["run_id"] == b1["run"] == decision["run"] == RUN_KEY
    g3, part = b1["g3"], b1[G3_SET]
    assert g3["set"] == G3_SET
    against, scores = part["against_b1"]["ours"], part["scores"]
    for score in g3["judged_on"]:
        assert standing(against[score]) == g3[score], f"{label}: G3 {score} reads {g3[score]}"
    conditions = robust["conditions"]  # notebook 21 calls the gap's first term as_is_masked, this one as_is_same_photos
    same_photos = conditions["swapped"]["as_is_same_photos"]["top1"]
    assert conditions["as_is_masked"]["top1"] == same_photos, f"{label}: as_is_masked and as_is_same_photos differ"
    gap, species = swap_gap(robust), counts[RUN_KEY][G2_LEVEL]
    fit, verdict = decision["recall_bound_fit"], decision["verdict_metrics"]
    low, high = decision["intervals"]["who_dangerous_recall"]
    row = {"run": label, "kernel": manifest["kernel"], "front_end": config["front_end"],
           "mask_source": config["mask_source"], "detail_crop": config["detail_crop"],
           "best_epoch": final["run"]["best_epoch"], "epochs": final["run"]["epochs"],
           "train_photos": final["run"]["train_photos"], "top1": final["head"]["top1"],
           "top3": final["head"]["top3"], "macro_f1": final["head"]["macro_f1"],
           "top1_same_photos": robust["conditions"]["swapped"]["as_is_same_photos"]["top1"],
           "top1_swapped": robust["conditions"]["swapped"]["top1"], "swap_photos": robust["photos"]["swapped"],
           "swap_gap": gap, "g1_mark": g1_mark, "g1_passed": gap <= g1_mark,
           "g2_species": species, "g2_measurable": counts[RUN_KEY]["measurable"], "g2_mark": G2_SPECIES,
           "g2_f1": G2_F1, "g2_passed": species >= G2_SPECIES, "b1_species": b1["shared_classes"], "g3_photos": part["photos"]}
    for score in g3["judged_on"]:
        row |= {f"g3_{score}_ours": scores["ours"][score], f"g3_{score}_b1": scores["B1"][score],
                f"g3_{score}_diff": against[score]["diff"], f"g3_{score}_low": against[score]["low"],
                f"g3_{score}_high": against[score]["high"], f"g3_{score}_standing": g3[score]}
    row["g3_passed"] = g3["passed"]
    for score in g3["judged_on"]:
        closed = scores["ours, closed set"][score]
        row |= {f"closed_{score}": closed, f"closed_{score}_minus_b1": closed - scores["B1"][score]}
    row |= {"cal_who_lower_bound": fit["lower_bound"], "safety_mark": fit["required"],
            "safety_passed": fit["lower_bound"] >= fit["required"],
            "val_who_point": verdict["who_dangerous_recall"], "val_who_low": low, "val_who_high": high,
            "val_false_alarm": verdict["false_alarm"], "source": named(folder)}
    return row


runs = pd.DataFrame([full_run_row(label) for label in FULL_RUNS])
runs["goals_passed"] = runs[["g1_passed", "g2_passed", "g3_passed"]].sum(axis=1)
runs["eligible"] = runs["safety_passed"]
ranked = runs[runs["eligible"]].sort_values(["goals_passed", "swap_gap", "macro_f1"],
                                            ascending=[False, True, False])
assert len(ranked), "no full run keeps the safety bound"
picked = ranked["run"].iloc[0]
assert picked == SHIPPED, f"the shipping rule picks {picked}, not {SHIPPED}: stop and report"
runs["shipped"] = runs["run"].eq(picked)
save(runs, "full_runs")
print("eligible (calibration WHO lower bound at the safety mark):", list(runs.loc[runs.eligible, "run"]))
print("goals passed of G1 to G3:", dict(zip(runs["run"], runs["goals_passed"])))
top = ranked[ranked["goals_passed"] == ranked["goals_passed"].iloc[0]]
if len(top) > 1:
    print("tie between", list(top["run"]), "on goals; swap gap:",
          {run: round(gap, 4) for run, gap in zip(top["run"], top["swap_gap"])})
print("shipped:", picked)
runs.set_index("run").drop(columns=["source", "kernel"]).T

wrote results/m4/summary/full_runs.csv: 3 rows
eligible (calibration WHO lower bound at the safety mark): ['A1', 'H2', 'A3']
goals passed of G1 to G3: {'A1': 2, 'H2': 1, 'A3': 2}
tie between ['A3', 'A1'] on goals; swap gap: {'A3': 0.0519, 'A1': 0.0754}
shipped: A3


run,A1,H2,A3
front_end,F1,F1,F1m
mask_source,sam,sam,predicted
detail_crop,False,True,False
best_epoch,28,24,24
epochs,30,30,30
train_photos,49782,49782,49782
top1,0.677,0.653,0.664
top3,0.831,0.808,0.822
macro_f1,0.481,0.437,0.481
top1_same_photos,0.685,0.661,0.682


## 3. Against B1

Every run against B1 (Bolon 2022) on the photos of B1's species: all validation photos, and the 2022
upload slice, which B1 cannot have trained on. G3 is judged on the slice, on top-1 and on macro-F1 over
the photos of those species. Also kept: macro-F1 by Milestone 3's whole-slice definition; our model
restricted to B1's species (the closed set, reported and not judged; the files hold no interval for
it); Milestone 3's final model; and a control with our predictions shuffled, which must lose to B1.

In [4]:
B1_SETS = ("full_validation", "upload_2022_slice")
B1_SCORES = ("top1", "macro_f1", "macro_f1_m3_definition")
SHUFFLED = "ours, predictions shuffled"
rows, references = [], {}
for label in FULL_RUNS:
    b1 = read_json(f"full/{label}/b1_62.json")
    for check in b1["b1_check"].values():
        assert check["photos"] == check["recorded photos"] and np.isclose(check["top-1"], check["recorded top-1"])
        assert np.isclose(check["macro-F1, whole slice"], check["recorded macro-F1"])
    for set_name in B1_SETS:
        part = b1[set_name]
        for score in B1_SCORES:
            ours, m3 = part["against_b1"]["ours"][score], part["against_b1"][M3_KEY][score]
            scores = part["scores"]
            reference = (scores["B1"][score], scores[M3_KEY][score], part["photos"])
            assert np.allclose(references.setdefault((set_name, score), reference), reference), "B1 or M3 differs by run"
            rows.append({"run": label, "set": set_name, "score": score, "photos": part["photos"],
                         "photographers": part["photographers"], "ours": scores["ours"][score],
                         "b1": scores["B1"][score], "ours_minus_b1": ours["diff"], "low": ours["low"],
                         "high": ours["high"], "standing": standing(ours),
                         "judged": set_name == b1["g3"]["set"] and score in b1["g3"]["judged_on"],
                         "ours_closed_set": scores["ours, closed set"][score],
                         "closed_minus_b1": scores["ours, closed set"][score] - scores["B1"][score],
                         "m3_final": scores[M3_KEY][score], "m3_minus_b1": m3["diff"], "m3_low": m3["low"],
                         "m3_high": m3["high"], "shuffled_minus_b1_high": part["against_b1"][SHUFFLED][score]["high"],
                         "source": named(f"full/{label}/b1_62.json")})
b1_table = save(pd.DataFrame(rows), "b1")
assert (b1_table["shuffled_minus_b1_high"] < 0).all(), "the shuffled control does not lose to B1"
b1_table[b1_table["judged"]].drop(columns=["source", "judged"]).set_index(["run", "score"])

wrote results/m4/summary/b1.csv: 18 rows


set  photos  photographers   ours     b1  ours_minus_b1    low       high standing  ours_closed_set  closed_minus_b1  m3_final  m3_minus_b1  m3_low  m3_high  shuffled_minus_b1_high
run score                                                                                                                                                                                                       
A1  top1      upload_2022_slice   11931           1767  0.723  0.788         -0.065 -0.083 -4.719e-02    lower            0.775       -1.349e-02     0.679       -0.109  -0.123   -0.094                  -0.741
    macro_f1  upload_2022_slice   11931           1767  0.713  0.719         -0.006 -0.024  1.647e-02     tied            0.718       -2.998e-04     0.594       -0.125  -0.140   -0.098                  -0.672
H2  top1      upload_2022_slice   11931           1767  0.705  0.788         -0.083 -0.101 -6.550e-02    lower            0.736       -5.247e-02     0.679       -0.109  -0.123   -0.094                  -0.739
    macro_f1  upload_2022_slice   11931           1767  0.666  0.719         -0.053 -0.071 -2.832e-02    lower            0.672       -4.678e-02     0.594       -0.125  -0.140   -0.098                  -0.672
A3  top1      upload_2022_slice   11931           1767  0.707  0.788         -0.081 -0.100 -6.380e-02    lower            0.760       -2.783e-02     0.679       -0.109  -0.123   -0.094                  -0.741
    macro_f1  upload_2022_slice   11931           1767  0.694  0.719         -0.025 -0.043 -6.914e-04    lower            0.701       -1.800e-02     0.594       -0.125  -0.140   -0.098                  -0.672

## 4. Species reached

How many of the measurable species (5 or more validation photos) reach each F1, for Milestone 3's final
model and the three full runs, from each run's `per_class_counts.json`. Macro-F1 is also split by how
many training photos a species had in that run; for Milestone 3 only its overall macro-F1 is in these
files (A0's row of the arm table).

In [5]:
BANDS = {"<30": "macro_f1_under_30_train", "30-99": "macro_f1_30_to_99_train",
         "100+": "macro_f1_100_plus_train"}
m3_macro_f1 = arms.set_index("arm").loc["A0", "macro_f1"]
rows, m3_counts = [], None
for label in FULL_RUNS:
    path = f"full/{label}/per_class_counts.json"
    counts, head = read_json(path), read_json(f"full/{label}/final_metrics.json")["head"]
    if m3_counts is None:
        m3_counts = counts[M3_KEY]
        rows.append({"run": M3_KEY, **m3_counts, "macro_f1": m3_macro_f1,
                     "source": named(path) + "; " + named("arms/arms.csv")})
    assert counts[M3_KEY] == m3_counts, f"{label} holds another M3 row"
    rows.append({"run": label, **counts[RUN_KEY], "macro_f1": head["macro_f1"],
                 **{BANDS[band]: value for band, value in head["macro_f1_band"].items()},
                 "pooled_recall_rare": head["pooled_recall_rare"],
                 "source": named(path) + "; " + named(f"full/{label}/final_metrics.json")})
classes = save(pd.DataFrame(rows), "classes")
classes.drop(columns="source").set_index("run")

wrote results/m4/summary/classes.csv: 4 rows


,measurable,f1_at_least_0.3,f1_at_least_0.5,f1_at_least_0.7,median_f1,f1_zero,macro_f1,macro_f1_under_30_train,macro_f1_30_to_99_train,macro_f1_100_plus_train,pooled_recall_rare
run,,,,,,,,,,,
M3 final,158,85,56,17,0.336,18,0.367,NaN,NaN,NaN,NaN
A1,158,114,87,47,0.552,23,0.481,0.186,0.370,0.581,0.319
H2,158,106,74,34,0.472,27,0.437,0.222,0.317,0.531,0.319
A3,158,116,87,47,0.534,20,0.481,0.226,0.376,0.572,0.319


## 5. The no-snake gate

Notebook 20 retrained the detector and the segmenter with non-snake photos and compared three gates (the
box, the mask, or both must pass) on the old and the new models. Each cut-off was fitted on calibration
snakes to keep the design's share of them, and the non-snake validation photos picked the gate. The
non-snake test split was then scored once, with the picked gate and, for the record, three other rules.

The tables below: every gate on calibration, validation and (where scored) test; false snakes by
category of non-snake photo; snake recall for groups of unusual snakes; and two checks on the front end
(Milestone 3's numbers reproduced before anything was retrained, and how far the new detector's boxes
are from the old one's).

In [6]:
gates = pd.read_csv(RESULTS / "gate/gates.csv")
gate, test = read_json("gate/gate.json"), read_json("gate/non_snake_test.json")
rules = test["rules"]
names = gates["front_end"] + " " + gates["rule"]
assert set(rules) <= set(names), "a test rule is not in gates.csv"
for name, rule in rules.items():
    fitted = json.loads(gates.loc[names.eq(name), "thresholds"].iloc[0])
    assert all(np.isclose(rule["thresholds"][key], value, atol=1e-6) for key, value in fitted.items()), name
picked_gate = f"M4 {gate['rule']}"
assert gates.loc[gates["picked"], "rule"].tolist() == [gate["rule"]] and test["g4"]["gate"] == picked_gate
assert rules[picked_gate]["picked"] and test["scored_once"] and test["split"] == "test"
gate_table = save(gates.assign(
    fpr_nonsnake_test=names.map(lambda name: rules.get(name, {}).get("false_snake_rate", np.nan)),
    called_a_snake_test=names.map(lambda name: rules.get(name, {}).get("called_a_snake")).astype("Int64"),
    n_nonsnake_test=test["photos"], keep=gate["keep"],
    source=named("gate/gates.csv") + "; " + named("gate/non_snake_test.json")), "gates")
g4 = test["g4"]
g4_passed = bool(g4["false_snake_rate"] <= g4["goal"] and g4["val_snake_recall"] >= gate["keep"])
print(f"picked gate {picked_gate}: test false snakes {g4['false_snake_rate']:.4f} of {test['photos']} "
      f"(goal {g4['goal']}), validation snake recall {g4['val_snake_recall']:.4f} (keep {gate['keep']}): "
      f"{'passed' if g4_passed else 'not passed'}; the file says passed={g4['passed']}")
gate_table.drop(columns="source").set_index(["front_end", "rule"])

wrote results/m4/summary/gates.csv: 6 rows
picked gate M4 both: test false snakes 0.2833 of 600 (goal 0.05), validation snake recall 0.9828 (keep 0.99): not passed; the file says passed=False


thresholds  recall_calibration  fpr_nonsnake_val  recall_snake_val  n_calibration  n_nonsnake_val  n_snake_val  meets_g4_on_val  picked  fpr_nonsnake_test  \
front_end rule                                                                                                                                                                                                 
M3        box                         {"box_conf": 0.101952}                0.99             0.953             0.991           7999             600        17151            False   False              0.957   
          mask                       {"mask_area": 0.005922}                0.99             0.987             0.995           7999             600        17151            False   False                NaN   
          both  {"box_conf": 0.087618, "mask_area": 0.00437}                0.99             0.962             0.992           7999             600        17151            False   False                NaN   
M4        box                         {"box_conf": 0.085111}                0.99             0.392             0.990           7999             600        17151            False   False              0.407   
          mask                        {"mask_area": 4.8e-05}                0.98             0.280             0.983           7999             600        17151            False   False              0.283   
          both  {"box_conf": 0.004993, "mask_area": 4.8e-05}                0.98             0.273             0.983           7999             600        17151            False    True              0.283   

                called_a_snake_test  n_nonsnake_test  keep  
front_end rule                                              
M3        box                   574              600  0.99  
          mask                 <NA>              600  0.99  
          both                 <NA>              600  0.99  
M4        box                   244              600  0.99  
          mask                  170              600  0.99  
          both                  170              600  0.99

In [7]:
def matching_rule(column, weights, front_end):
    """The gate whose validation false-snake rate a by-category column adds up to."""
    total = (column * weights).sum() / weights.sum()
    found = gates[gates["front_end"].eq(front_end) & np.isclose(gates["fpr_nonsnake_val"], total)]
    assert len(found) == 1, f"{front_end}: no single gate adds up to {total}"
    return found["rule"].iloc[0]


by_category = pd.read_csv(RESULTS / "gate/fpr_by_category.csv")
m3_rule = matching_rule(by_category["M3 called a snake"], by_category["photos"], "M3")
m4_rule = matching_rule(by_category["M4 called a snake"], by_category["photos"], "M4")
assert m4_rule == gate["rule"]
category = by_category["category"]
assert np.allclose(by_category["M4 called a snake"].round(4), category.map(test["val_by_category_for_the_gate"]))
categories = pd.DataFrame({
    "category": category, "lookalike": category.isin(LOOKALIKES), "photos_val": by_category["photos"],
    f"m3_{m3_rule}_val": by_category["M3 called a snake"], f"m4_{m4_rule}_val": by_category["M4 called a snake"],
    "photos_test": category.map(test["photos_per_category"]),
    "m3_box_test": category.map(rules[BEFORE_GATE]["by_category"]),
    f"m4_{m4_rule}_test": category.map(rules[picked_gate]["by_category"]),
    "source": named("gate/fpr_by_category.csv") + "; " + named("gate/non_snake_test.json")})
weighted = (categories[f"m4_{m4_rule}_test"] * categories["photos_test"]).sum() / categories["photos_test"].sum()
assert abs(weighted - g4["false_snake_rate"]) < 1e-3, "the test categories do not add up to the test rate"
save(categories, "gate_categories")
categories.drop(columns="source").set_index("category")

wrote results/m4/summary/gate_categories.csv: 11 rows


,lookalike,photos_val,m3_both_val,m4_both_val,photos_test,m3_box_test,m4_both_test
category,,,,,,,
caecilian,True,20,1.000,1.000,20,1.000,0.950
cat,False,40,0.975,0.075,40,1.000,0.025
dog,False,40,0.950,0.050,40,0.975,0.050
earthworm,True,20,1.000,1.000,20,1.000,1.000
eel,True,40,0.975,0.950,40,0.975,0.975
hand_arm,False,60,0.983,0.067,60,0.900,0.083
legless_lizard,True,20,1.000,1.000,20,1.000,1.000
other_animal,False,60,0.983,0.417,60,0.983,0.433
person,False,180,0.928,0.083,180,0.933,0.089


In [8]:
ALL_SNAKES = "all val snakes"
groups = pd.read_csv(RESULTS / "gate/lookalike_recall.csv")
everything = groups.set_index("group").loc[ALL_SNAKES]
for row in gates.itertuples():
    assert np.isclose(everything[f"{row.front_end} {row.rule}"], row.recall_snake_val), "all val snakes differ"
save(groups.assign(source=named("gate/lookalike_recall.csv")), "gate_snake_groups")
reproduction, boxes = read_json("gate/m3_reproduction.json"), read_json("gate/box_agreement.json")
tolerance = reproduction["tolerances"]
checks = pd.DataFrame([
    {"check": "M3 detector: box at the floor in both runs or neither", "value": reproduction["box_agreement"],
     "mark": tolerance["box_agreement"], "rule": "at least", "photos": reproduction["photos"]},
    {"check": "M3 segmenter: mask agreement", "value": reproduction["mask_agreement"],
     "mark": tolerance["mask_agreement"], "rule": "at least", "photos": reproduction["photos"]},
    {"check": "M3 cut-off refitted minus M3's", "value": reproduction["tau_snake_refit"] - reproduction["tau_snake_m3"],
     "mark": tolerance["tau_gap"], "rule": "within", "photos": reproduction["photos"]},
    {"check": "M4 against M3 detector boxes: median IoU", "value": boxes["median_iou"], "mark": np.nan,
     "rule": "reported", "photos": boxes["photos"]},
    {"check": "M4 against M3 detector boxes: share at IoU 0.5 or more", "value": boxes["share_iou_at_least_0_5"],
     "mark": np.nan, "rule": "reported", "photos": boxes["photos"]}])
checks["source"] = [named("gate/m3_reproduction.json")] * 3 + [named("gate/box_agreement.json")] * 2
save(checks, "gate_checks")
display(groups.set_index("group"))
checks.drop(columns="source")

wrote results/m4/summary/gate_snake_groups.csv: 6 rows
wrote results/m4/summary/gate_checks.csv: 5 rows


,photos,M3 box,M3 mask,M3 both,M4 box,M4 mask,M4 both
group,,,,,,,
sea snakes and sea kraits,715,0.999,0.987,0.992,0.993,0.969,0.969
other water snakes,318,0.994,0.994,0.991,0.991,0.991,0.991
blind snakes and shieldtails,1763,0.997,0.993,0.996,0.995,0.974,0.974
vine snakes and bronzebacks,1959,0.989,0.995,0.991,0.989,0.987,0.987
all other snakes,12396,0.990,0.995,0.991,0.989,0.984,0.984
all val snakes,17151,0.991,0.995,0.992,0.990,0.983,0.983


,check,value,mark,rule,photos
0,M3 detector: box at the floor in both runs or ...,1.000e+00,0.99,at least,25150
1,M3 segmenter: mask agreement,1.000e+00,0.99,at least,25150
2,M3 cut-off refitted minus M3's,-4.789e-05,0.01,within,25150
3,M4 against M3 detector boxes: median IoU,9.420e-01,NaN,reported,17082
4,M4 against M3 detector boxes: share at IoU 0.5...,9.090e-01,NaN,reported,17082


## 6. Head crops

Two ways of letting the model look closer at the head were tried. Approach 1 needed a head detector, so
notebook 22 first asked three open-vocabulary detectors for "snake head" and "snake tail" boxes on the
same training photos; each box was judged on a contact sheet (the verdicts are in `heads/pilot.csv` and
`heads/pilot_retry.csv`), and a part had to have enough of its boxes right before any head detector was
built. Approach 2 (notebook 23) needs no labels: the model crops the spot its class activation map rates
highest, looks again and averages the two looks. It was trained on the
subset on top of A1 and compared with A1's own row, then trained in full as H2.

In [9]:
pilot = pd.read_csv(RESULTS / "heads/pilot.csv").assign(model="dino_tiny", prompting="both parts in one prompt")
retry = pd.read_csv(RESULTS / "heads/pilot_retry.csv").assign(prompting="one prompt per part")
attempts = pd.concat([pilot, retry], ignore_index=True)
assert attempts["right"].notna().all() and attempts["right"].isin([True, False]).all(), "a box has no verdict"
head_pilot = (attempts.assign(drawn=attempts["box"].ne("none"))
              .groupby(["model", "prompting", "part"], sort=False)
              .agg(photos=("path", "nunique"), boxes_drawn=("drawn", "sum"), right=("right", "sum"))
              .reset_index())
head_pilot = save(head_pilot.assign(pass_at=PILOT_PASS_AT, passed=head_pilot["right"] >= PILOT_PASS_AT,
                                    source=head_pilot["prompting"].map(
                                        {"both parts in one prompt": named("heads/pilot.csv"),
                                         "one prompt per part": named("heads/pilot_retry.csv")})), "head_pilot")
print("parts passing:", int(head_pilot["passed"].sum()), "of", len(head_pilot))
head_pilot.drop(columns="source")

wrote results/m4/summary/head_pilot.csv: 8 rows
parts passing: 0 of 8


,model,prompting,part,photos,boxes_drawn,right,pass_at,passed
0,dino_tiny,both parts in one prompt,head,50,47,4,40,False
1,dino_tiny,both parts in one prompt,tail,50,48,1,40,False
2,dino_tiny,one prompt per part,head,50,49,4,40,False
3,dino_tiny,one prompt per part,tail,50,49,1,40,False
4,dino_base,one prompt per part,head,50,50,3,40,False
5,dino_base,one prompt per part,tail,50,50,0,40,False
6,owlv2,one prompt per part,head,50,42,4,40,False
7,owlv2,one prompt per part,tail,50,45,3,40,False


In [10]:
h2, a1 = read_json("heads/M4_H2.json"), read_json("arms/M4_A1/arm_row.json")
a1_final = read_json("arms/M4_A1/final_metrics.json")
assert h2["winner"] == a1["run"] and h2["bank_sha256"] == a1["bank_sha256"] and h2["swap_photos"] == a1["swap_photos"]
SUBSET_MEASURES = {"macro_f1": "macro_f1", "who_recall_dr0": "who_recall_dr0", "top1": "top1_photos_with_mask",
                   "top1_swapped": "top1_swapped", "top1_blanked": "top1_blanked", "gap": "swap_gap"}
FULL_MEASURES = ("top1", "macro_f1", "swap_gap", "g2_species", "cal_who_lower_bound", "val_who_point")
H2_FILE, A1_FILE = named("heads/M4_H2.json"), named("arms/M4_A1/arm_row.json")
rows = [{"scale": "subset", "measure": name, "A1": a1[key], "H2": h2[key], "source_A1": A1_FILE,
         "source_H2": H2_FILE} for key, name in SUBSET_MEASURES.items()]
rows.append({"scale": "subset", "measure": "top3", "A1": a1_final["head"]["top3"], "H2": h2["top3"],
             "source_A1": named("arms/M4_A1/final_metrics.json"), "source_H2": H2_FILE})
by_run = runs.set_index("run")
rows += [{"scale": "full", "measure": name, "A1": by_run.loc["A1", name], "H2": by_run.loc["H2", name],
          "source_A1": named("full/A1"), "source_H2": named("full/H2")} for name in FULL_MEASURES]
laptop = h2["laptop_ms_per_frame"]
rows.append({"scale": "laptop", "measure": "ms_per_frame", "A1": laptop["one look"],
             "H2": laptop["with the detail crop"], "source_A1": H2_FILE, "source_H2": H2_FILE})
rows += [{"scale": "subset", "measure": f"detail_box_{key}", "A1": np.nan, "H2": h2["detail_boxes"][key],
          "source_A1": "", "source_H2": H2_FILE} for key in ("median_area", "whole_photo_share", "corner_share")]
heads = pd.DataFrame(rows)
heads.insert(4, "difference", heads["H2"] - heads["A1"])
save(heads, "heads")
heads.drop(columns=["source_A1", "source_H2"])

wrote results/m4/summary/heads.csv: 17 rows


,scale,measure,A1,H2,difference
0,subset,macro_f1,0.372,0.383,1.079e-02
1,subset,who_recall_dr0,0.945,0.949,3.754e-03
2,subset,top1_photos_with_mask,0.631,0.634,3.129e-03
3,subset,top1_swapped,0.538,0.541,3.255e-03
4,subset,top1_blanked,0.140,0.116,-2.460e-02
5,subset,swap_gap,0.093,0.093,-1.252e-04
6,subset,top3,0.764,0.768,3.906e-03
7,full,top1,0.677,0.653,-2.321e-02
8,full,macro_f1,0.481,0.437,-4.463e-02
9,full,swap_gap,0.075,0.080,4.694e-03


## 7. Video

Notebook 24 ran two rules on the same clips: Milestone 3's vote over the last frames (`vote`) and the
new lock (`lock`). Its first run used the Milestone 3 model; the rerun on the shipped model writes
`video/A3/`. Each table counts, per kind of clip and split, how often the shown species changes after
the first answer (over the clips that answer), how long the first answer takes, and how many final answers are right. Species are
scored only on clips whose page names the species (`stated` or `category`), not on the `inferred`
ones. G5 is read on the real test clips under the lock. It is judged on the clips that answer: a clip
that never shows a species is marked `no answer`, is not within G5, and is counted in the row; if no
real test clip answers, G5 is not met.

In [11]:
VIDEO = RESULTS / "video"


clips = pd.read_csv(VIDEO / "clips.csv")
folders = {model: VIDEO / model for model in VIDEO_MODELS}
assert all(folder.is_dir() for folder in folders.values()), "a model's video results are missing"
summaries, per_clip = [], []
for model, folder in folders.items():
    lock = pd.read_csv(folder / "video_lock.csv")
    lock["final_right"] = lock["final_right"].astype("boolean")
    within = np.where(lock["changes_after_first_answer"] <= G5_MAX_CHANGES, "yes", "no")
    lock["within_g5"] = np.where(lock["answered"], within, NO_ANSWER)
    lock["changes_if_answered"] = lock["changes_after_first_answer"].where(lock["answered"])
    source = named((folder / "video_lock.csv").relative_to(RESULTS).as_posix())
    print(f"{model}: {source}")
    summaries.append(lock.groupby(["kind", "split", "method"]).agg(
        clips=("clip", "size"), species_scored=("final_right", "count"), final_right=("final_right", "sum"),
        median_changes=("changes_if_answered", "median"), max_changes=("changes_if_answered", "max"),
        within_g5=("within_g5", lambda found: int(found.eq("yes").sum())), answered=("answered", "sum"),
        never_answered=("answered", lambda found: int((~found).sum())),
        median_first_answer_s=("first_answer_s", "median")).reset_index().assign(model=model, source=source))
    real = lock[lock["kind"].eq("real")].merge(clips[["clip", "species_source"]], on="clip", how="left",
                                               validate="many_to_one")
    assert real["species_source"].notna().all(), "a real clip is missing from clips.csv"
    per_clip.append(real[["clip", "split", "species", "species_source", "method", "snake_frames",
                          "changes_after_first_answer", "answered", "within_g5", "first_answer_s",
                          "final_shown", "final_right"]].assign(model=model, source=source))
video_table = save(pd.concat(summaries, ignore_index=True), "video")
video_clips = save(pd.concat(per_clip, ignore_index=True), "video_clips")
display(video_table.drop(columns="source").set_index(["model", "kind", "split", "method"]))
video_clips[video_clips["split"].eq("test")].drop(columns="source").set_index(["model", "method", "clip"])

M3: results/m4/video/M3/video_lock.csv
A3: results/m4/video/A3/video_lock.csv
wrote results/m4/summary/video.csv: 16 rows
wrote results/m4/summary/video_clips.csv: 40 rows


clips  species_scored  final_right  median_changes  max_changes  within_g5  answered  never_answered  median_first_answer_s
model kind                 split method                                                                                                                             
M3    real                 dev   lock        4               4            1             1.0          4.0          2         3               1                   1.70
                                 vote        4               4            3            10.0         17.0          0         4               0                   1.05
                           test  lock        6               4            1             0.0          0.0          5         5               1                   1.00
                                 vote        6               4            1             0.5          9.0          4         6               0                   0.40
      synthetic_random     dev   lock       20              20            2             0.0          2.0          8         9              11                   1.00
                                 vote       20              20            3             2.5          7.0          8        20               0                   0.65
      synthetic_who_listed dev   lock       26              26            7             0.0          1.0         16        16              10                   1.45
                                 vote       26              26            6             1.0          7.0         14        25               1                   0.70
A3    real                 dev   lock        4               4            3             0.0          0.0          4         4               0                   0.80
                                 vote        4               4            3             0.5          6.0          3         4               0                   0.85
                           test  lock        6               4            1             0.0          2.0          4         5               1                   0.50
                                 vote        6               4            1             3.0         17.0          1         6               0                   0.40
      synthetic_random     dev   lock       20              20            5             0.0          1.0         14        14               6                   0.50
                                 vote       20              20            2             0.0          6.0         13        20               0                   0.65
      synthetic_who_listed dev   lock       26              26            8             0.0          2.0         17        18               8                   0.50
                                 vote       26              26            5             1.0          7.0         14        25               1                   0.50

split                species species_source  snake_frames  changes_after_first_answer  answered  within_g5  first_answer_s                   final_shown  final_right
model method clip                                                                                                                                                                                                 
M3    vote   naja_naja_ditmars_1915.mp4       test              Naja naja         stated           301                           0      True        yes             0.4                           NaN        False
      lock   naja_naja_ditmars_1915.mp4       test              Naja naja         stated           301                           0      True        yes             1.0               Fowlea piscator        False
      vote   gonyosoma_oxycephalum_mulu.webm  test  Gonyosoma oxycephalum         stated            68                           0      True        yes             0.4         Gonyosoma oxycephalum         True
      lock   gonyosoma_oxycephalum_mulu.webm  test  Gonyosoma oxycephalum         stated            68                           0      True        yes             1.0         Gonyosoma oxycephalum         True
      vote   ahaetulla_borealis_kerala.webm   test     Ahaetulla borealis       category            53                           1      True        yes            22.8  Trimeresurus septentrionalis        False
      lock   ahaetulla_borealis_kerala.webm   test     Ahaetulla borealis       category            53                           0      True        yes            25.0             Ahaetulla prasina        False
      vote   ptyas_mucosa_swimming.webm       test           Ptyas mucosa       category           268                           9      True         no             0.8                           NaN        False
      lock   ptyas_mucosa_swimming.webm       test           Ptyas mucosa       category           268                           0     False  no answer             NaN                           NaN        False
      vote   naja_cobra_2017.webm             test              Naja naja       inferred           301                           2      True         no             0.4                     Naja naja         <NA>
      lock   naja_cobra_2017.webm             test              Naja naja       inferred           301                           0      True        yes             1.0                     Naja naja         <NA>
      vote   naja_jaipur_charmer.webm         test              Naja naja       inferred           158                           0      True        yes             0.4            Ophiophagus hannah         <NA>
      lock   naja_jaipur_charmer.webm         test              Naja naja       inferred           158                           0      True        yes             1.0            Ophiophagus hannah         <NA>
A3    vote   naja_naja_ditmars_1915.mp4       test              Naja naja         stated           301                          12      True         no             0.4               Fowlea piscator        False
      lock   naja_naja_ditmars_1915.mp4       test              Naja naja         stated           301                           2      True         no             0.5               Fowlea piscator        False
      vote   gonyosoma_oxycephalum_mulu.webm  test  Gonyosoma oxycephalum         stated            68                           0      True        yes             0.4         Gonyosoma oxycephalum         True
      lock   gonyosoma_oxycephalum_mulu.webm  test  Gonyosoma oxycephalum         stated            68                           0      True        yes             0.5         Gonyosoma oxycephalum         True
      vote   ahaetulla_borealis_kerala.webm   test     Ahaetulla borealis       category           301                          17      True         no             0.4                           NaN        False
      lock   ahaetulla_borealis_kera

### Tuning, negative controls and the front-end shift

The window and margin tuning of each run (`margin_tuning.csv`), next to `video.POOL_FRAMES`, the window
the code uses by default (read from `src/serpentguard/video.py`). The negative-control clips (a
chameleon, an eagle, a river) come from each run's saved notebook 24, its negative-control table: the
Milestone 3 run's executed copy in `video/executed/`, and the shipped model's run in
`notebooks/24_video_lock.ipynb`. The river is the Milestone 3 detector's false alarm, so the two clean
negatives are the chameleon and the eagle. Last, the front-end shift: the shipped model's top-1 on the
same validation photos through the crops it was trained on and through the app's front end; the cell
recounts every number from the file's per-photo rows.

In [12]:
NOTEBOOK_24 = {"M3": (RESULTS / "video/executed/24_video_lock_M3.ipynb", named("video/executed/24_video_lock_M3.ipynb")),
               SHIPPED: (Path("24_video_lock.ipynb"), "notebooks/24_video_lock.ipynb")}
NEGATIVES_CELL = "negatives = pd.DataFrame(rows)"  # the code that makes notebook 24's negative-control table
NOT_CLEAN = "negative_river_wyoming.webm"  # notebook 24: the Milestone 3 detector's false alarm, not a clean negative
VIDEO_CODE, VIDEO_CODE_NAMED = Path("../src/serpentguard/video.py"), "src/serpentguard/video.py"


def pool_frames():
    """video.POOL_FRAMES, read from the source without importing the package."""
    tree = ast.parse(VIDEO_CODE.read_text())
    return next(node.value.value for node in tree.body if isinstance(node, ast.Assign)
                and any(getattr(target, "id", None) == "POOL_FRAMES" for target in node.targets))


def html_rows(html):
    """The rows of a pandas HTML table as text, header first, index column dropped."""
    table = ElementTree.fromstring(html[html.index("<table"):html.index("</table>") + len("</table>")])
    return [[cell.text or "" for cell in row][1:] for row in table.iter("tr")]


def negative_controls(model):
    """Notebook 24's negative-control table from one model's saved run."""
    path, _ = NOTEBOOK_24[model]
    run = json.loads(path.read_text())
    printed = "".join(text for cell in run["cells"] if cell["cell_type"] == "code"
                      for output in cell["outputs"] for text in output.get("text", []))
    assert model == "M3" or f"video/{model}" in printed, f"{path} did not run on {model}"
    cell = next(c for c in run["cells"] if c["cell_type"] == "code" and NEGATIVES_CELL in "".join(c["source"]))
    html = next("".join(o["data"]["text/html"]) for o in cell["outputs"] if "text/html" in o.get("data", {}))
    header, *rows = html_rows(html)
    return pd.DataFrame(rows, columns=header).astype({"frames": int, "snake_frames": int, "locked_frames": int})


POOL_FRAMES = pool_frames()
negatives_list = pd.read_csv(VIDEO / "negatives.csv")
tuning, negatives = [], []
for model, folder in folders.items():
    grid = pd.read_csv(folder / "margin_tuning.csv").drop(columns="note")
    tuning.append(grid.assign(model=model, pool_frames=POOL_FRAMES,
                              source=named((folder / "margin_tuning.csv").relative_to(RESULTS).as_posix())
                              + "; " + VIDEO_CODE_NAMED))
    if NOTEBOOK_24[model][0].exists():
        found = negative_controls(model)
        assert set(found["clip"]) == set(negatives_list["clip"]), f"{model}: other negative clips"
        negatives.append(found.merge(negatives_list[["clip", "content"]], on="clip", validate="one_to_one")
                         .assign(model=model, clean=found["clip"].ne(NOT_CLEAN), source=NOTEBOOK_24[model][1]))
video_tuning = save(pd.concat(tuning, ignore_index=True), "video_tuning")
video_negatives = save(pd.concat(negatives, ignore_index=True), "video_negatives")
chosen = video_tuning[video_tuning["chosen"]]
clean_count = int(video_negatives.drop_duplicates("clip")["clean"].sum())
for row in chosen.itertuples():
    grid = video_tuning[video_tuning["model"].eq(row.model)]
    print(f"{row.model}: chosen window {row.window}, margin {row.margin} (POOL_FRAMES {POOL_FRAMES}); "
          f"{int(grid['clean_negatives_locked'].eq(clean_count).sum())} of {len(grid)} pairs lock on all "
          f"{clean_count} clean negatives")
display(chosen.drop(columns="source").set_index("model"))
video_negatives.drop(columns="source").set_index(["clip", "model"])

wrote results/m4/summary/video_tuning.csv: 20 rows
wrote results/m4/summary/video_negatives.csv: 6 rows
M3: chosen window 10, margin 0.05 (POOL_FRAMES 10); 5 of 10 pairs lock on all 2 clean negatives
A3: chosen window 5, margin 0.05 (POOL_FRAMES 10); 10 of 10 pairs lock on all 2 clean negatives


,window,margin,median_changes,median_first_answer_s,never_answered,final_right,clean_negatives_locked,eligible,rule_met,chosen,pool_frames
model,,,,,,,,,,,
M3,10,0.05,0.0,6.20,22,10,0,True,False,True,10
A3,5,0.05,0.0,1.15,14,16,2,True,True,True,10


,,frames,snake_frames,locked_frames,verdicts_shown,content,clean,locked_to
clip,model,,,,,,,
negative_chameleon_1915.mp4,M3,301,301,0,[dangerous],"a chameleon, no snake",True,NaN
negative_eagle_goa.ogv,M3,189,184,0,[dangerous],"a crested serpent eagle in flight, no snake",True,NaN
negative_river_wyoming.webm,M3,301,151,234,[dangerous],"a river seen from a drone, no snake",False,NaN
negative_chameleon_1915.mp4,A3,301,294,297,[dangerous],"a chameleon, no snake",True,[Echis carinatus]
negative_eagle_goa.ogv,A3,189,82,147,[dangerous],"a crested serpent eagle in flight, no snake",True,"[Hydrophis platurus, Laticauda colubrina]"
negative_river_wyoming.webm,A3,301,84,8,[dangerous],"a river seen from a drone, no snake",False,[Python bivittatus]


In [13]:
SHIFT = VIDEO / SHIPPED / "front_end_shift.json"
SHIFT_KEYS = ("photos", "species", "saved_top1", "m4_top1", "m4_minus_saved", "agreement", "only_saved_right",
              "only_m4_right", "paired_exact_p", "gate_dropped", "gate_dropped_saved_right", "tolerance",
              "within_tolerance")
assert SHIFT.exists(), f"{SHIFT} is missing"
shift = json.loads(SHIFT.read_text())
photos, control = pd.DataFrame(shift["per_photo"]), shift["m3_unet_control"]
saved_right, m4_right = photos["saved"].eq(photos["true"]), photos["m4"].eq(photos["true"])
assert len(photos) == shift["photos"] and photos["true"].nunique() == shift["species"]
assert np.isclose(saved_right.mean(), shift["saved_top1"]) and np.isclose(m4_right.mean(), shift["m4_top1"])
assert np.isclose(shift["m4_top1"] - shift["saved_top1"], shift["m4_minus_saved"])
assert np.isclose(photos["saved"].eq(photos["m4"]).mean(), shift["agreement"])
assert (saved_right & ~m4_right).sum() == shift["only_saved_right"]
assert (m4_right & ~saved_right).sum() == shift["only_m4_right"]
assert np.isclose(binomtest(shift["only_m4_right"], shift["only_m4_right"] + shift["only_saved_right"]).pvalue,
                  shift["paired_exact_p"])
assert (~photos["gate_passed"]).sum() == shift["gate_dropped"]
assert np.isclose(photos["m3_unet_control"].eq(photos["true"]).mean(), control["top1"])
assert np.isclose(photos["m3_unet_control"].eq(photos["saved"]).mean(), control["agreement_with_saved"])
shift_table = save(pd.DataFrame([{
    "model": SHIPPED, **{key: shift[key] for key in SHIFT_KEYS},
    "m3_unet_control_top1": control["top1"], "m3_unet_control_agreement_with_saved": control["agreement_with_saved"],
    "saved_is": shift["saved"], "m4_is": shift["m4"], "m3_unet_control_is": shift["m3_unet_control_is"],
    "source": named(f"video/{SHIPPED}/front_end_shift.json")}]), "front_end_shift")
display(shift_table.drop(columns=["source", "saved_is", "m4_is", "m3_unet_control_is"]).T)

wrote results/m4/summary/front_end_shift.csv: 1 rows


,0
model,A3
photos,300
species,176
saved_top1,0.477
m4_top1,0.443
m4_minus_saved,-0.033
agreement,0.85
only_saved_right,16
only_m4_right,6
paired_exact_p,0.052


## 8. The goals, for the shipped model

One row per goal for the model the rule shipped, with the number, its mark, pass or fail, and the file
it comes from; Milestone 3's number where the files hold one.

In [14]:
ship = runs.set_index("run").loc[SHIPPED]
ship_dir = f"full/{SHIPPED}"
m3_decision = read_json(M3_DECISION)
against = read_json(f"{ship_dir}/b1_62.json")[G3_SET]["against_b1"]
m3_slice = against[M3_KEY]
g3_rule = read_json(f"{ship_dir}/b1_62.json")["g3"]["rule"]
no_snake = read_json(f"{ship_dir}/decision_metrics.json")["no_snake"]
gate_recall = gates.loc[gates["picked"], "recall_snake_val"].iloc[0]
sea_recall = groups.set_index("group").loc[SEA_SNAKES, picked_gate]
safety_front_end = (f"measured behind M3's box gate (box_conf at least {no_snake['tau_snake']}, which set "
                    f"{no_snake['val']} validation photos aside as no snake) on crops from M3's front end, not "
                    f"behind the app's M4 front end, whose gate ({picked_gate}) drops {1 - gate_recall:.2%} of "
                    f"validation snakes ({1 - sea_recall:.1%} of the {SEA_SNAKES}); WHO recall behind the M4 front "
                    "end is not measured yet")


def g3_text(row_get):
    return "; ".join(f"{score} {interval_text(*row_get(score))}" for score in ("top1", "macro_f1"))


def answered_clips(rows):
    """The clips G5 judges (those that answer), and whether every one of them stays within G5."""
    answered = rows[rows["answered"]]
    return answered, bool(len(answered)) and bool(answered["within_g5"].eq("yes").all())


def g5_rows():
    """G5 for the shipped model and Milestone 3's vote, from the real test clips."""
    test_clips = video_clips[video_clips["split"].eq("test")]
    m3 = test_clips[test_clips["model"].eq("M3") & test_clips["method"].eq("vote")]
    m3_answered, _ = answered_clips(m3)
    m3_value = m3_answered["changes_after_first_answer"].max() if len(m3_answered) else np.nan
    m3_detail = (f"M3 model, M3's vote: {len(m3_answered)} of {len(m3)} clips answered; changes per answered "
                 f"clip {m3_answered['changes_after_first_answer'].tolist()}" if len(m3) else "")
    m3_source = m3["source"].iloc[0] if len(m3) else ""
    a3 = test_clips[test_clips["model"].eq(SHIPPED) & test_clips["method"].eq("lock")]
    assert len(a3), f"{named(f'video/{SHIPPED}/video_lock.csv')} has no real test clip under the lock"
    answered, passed = answered_clips(a3)
    scored = a3["final_right"].dropna()
    counts = (f"{len(a3)} real test clips, {len(answered)} answered, {len(a3) - len(answered)} never answered "
              f"({NO_ANSWER}, not within G5)")
    if not len(answered):
        detail = f"{counts}; no real test clip answered, so G5 is not met"
    else:
        detail = (f"{counts}; changes per answered clip {answered['changes_after_first_answer'].tolist()}; "
                  f"median first answer {answered['first_answer_s'].median():.2f} s; final answer right on "
                  f"{int(scored.sum())} of {len(scored)} species-scored clips")
    return {"value": answered["changes_after_first_answer"].max() if len(answered) else np.nan,
            "passed": passed, "detail": detail,
            "source": a3["source"].iloc[0], "m3_value": m3_value, "m3_detail": m3_detail, "m3_source": m3_source}


goals = pd.DataFrame([
    {"goal": "G1", "measure": "swap gap: top-1 as is minus top-1 with the scenery swapped, same photos",
     "value": ship["swap_gap"], "mark": g1_mark, "rule": "at most", "passed": bool(ship["g1_passed"]),
     "detail": (f"top-1 {ship['top1_same_photos']:.4f} as is, {ship['top1_swapped']:.4f} swapped, "
                f"{ship['swap_photos']} photos; the mark is {G1_SHARE_OF_M3} of M3's gap"),
     "source": named(f"{ship_dir}/robustness_metrics.json"), "m3_value": m3_gap, "m3_detail": "",
     "m3_source": named(M3_ROBUSTNESS)},
    {"goal": "G2", "measure": f"measurable species at validation F1 {G2_F1} or above",
     "value": ship["g2_species"], "mark": G2_SPECIES, "rule": "at least", "passed": bool(ship["g2_passed"]),
     "detail": f"{ship['g2_species']} of {ship['g2_measurable']} measurable species",
     "source": named(f"{ship_dir}/per_class_counts.json"), "m3_value": m3_counts[G2_LEVEL], "m3_detail": "",
     "m3_source": named(f"{ship_dir}/per_class_counts.json")},
    {"goal": "G3", "measure": "ours minus B1 on B1's species, 2022 slice, 95% photographer-bootstrap interval",
     "value": np.nan, "mark": np.nan, "rule": g3_rule, "passed": bool(ship["g3_passed"]),
     "detail": (g3_text(lambda s: (ship[f"g3_{s}_diff"], ship[f"g3_{s}_low"], ship[f"g3_{s}_high"]))
                + f"; top1 {ship['g3_top1_standing']}, macro_f1 {ship['g3_macro_f1_standing']}; "
                  f"{ship['g3_photos']} photos; macro_f1 is per-species F1 over the photos of B1's species only. "
                  f"By M3's whole-slice definition (reported, not judged): "
                  f"{interval_text(*(against['ours'][M3_DEFINITION][k] for k in ('diff', 'low', 'high')))}, "
                  f"{standing(against['ours'][M3_DEFINITION])}"),
     "source": named(f"{ship_dir}/b1_62.json"), "m3_value": np.nan,
     "m3_detail": (g3_text(lambda s: (m3_slice[s]["diff"], m3_slice[s]["low"], m3_slice[s]["high"]))
                   + "; by M3's whole-slice definition: "
                   + interval_text(*(m3_slice[M3_DEFINITION][k] for k in ("diff", "low", "high")))),
     "m3_source": named(f"{ship_dir}/b1_62.json")},
    {"goal": "G4", "measure": "share of non-snake test photos called a snake, gate " + picked_gate,
     "value": g4["false_snake_rate"], "mark": g4["goal"], "rule": "at most", "passed": g4_passed,
     "detail": (f"{rules[picked_gate]['called_a_snake']} of {test['photos']} test photos; validation snake "
                f"recall {g4['val_snake_recall']:.4f}, mark at least {gate['keep']}"),
     "source": named("gate/non_snake_test.json"), "m3_value": rules[BEFORE_GATE]["false_snake_rate"],
     "m3_detail": f"M3's gate ({BEFORE_GATE}) on the same test photos", "m3_source": named("gate/non_snake_test.json")},
    {"goal": "G5", "measure": "most changes of the shown species after the first answer, per real test clip",
     "mark": G5_MAX_CHANGES, "rule": "at most", **g5_rows()},
    {"goal": "Safety", "measure": "WHO recall lower bound on calibration", "value": ship["cal_who_lower_bound"],
     "mark": ship["safety_mark"], "rule": "at least", "passed": bool(ship["safety_passed"]),
     "detail": (f"validation WHO recall {ship['val_who_point']:.4f} "
                f"({ship['val_who_low']:.4f} to {ship['val_who_high']:.4f}); {safety_front_end}"),
     "source": named(f"{ship_dir}/decision_metrics.json"),
     "m3_value": m3_decision["recall_bound_fit"]["lower_bound"], "m3_detail": "",
     "m3_source": named(M3_DECISION)}])
goals = save(goals[["goal", "measure", "value", "mark", "rule", "passed", "detail", "source", "m3_value",
                    "m3_detail", "m3_source"]], "goals")
goals.drop(columns=["measure", "m3_detail"])

wrote results/m4/summary/goals.csv: 6 rows


,goal,value,mark,rule,passed,detail,source,m3_value,m3_source
0,G1,0.052,0.084,at most,True,"top-1 0.6825 as is, 0.6306 swapped, 15977 phot...",results/m4/full/A3/robustness_metrics.json,0.168,results/m4/arms/A0_check/robustness_metrics.json
1,G2,87.000,80.000,at least,True,87 of 158 measurable species,results/m4/full/A3/per_class_counts.json,56.000,results/m4/full/A3/per_class_counts.json
2,G3,NaN,NaN,"top-1 and macro-F1 each higher than B1's, or t...",False,top1 -0.081 (-0.100 to -0.064); macro_f1 -0.02...,results/m4/full/A3/b1_62.json,NaN,results/m4/full/A3/b1_62.json
3,G4,0.283,0.050,at most,False,170 of 600 test photos; validation snake recal...,results/m4/gate/non_snake_test.json,0.957,results/m4/gate/non_snake_test.json
4,G5,2.000,1.000,at most,False,"6 real test clips, 5 answered, 1 never answere...",results/m4/video/A3/video_lock.csv,9.000,results/m4/video/M3/video_lock.csv
5,Safety,0.958,0.950,at least,True,validation WHO recall 0.9545 (0.9379 to 0.9693...,results/m4/full/A3/decision_metrics.json,0.952,results/m4/arms/A0_check/decision_metrics.json


## 9. Checks

Every source a table names must exist, the shares must lie between 0 and 1, and the numbers each goal
reads must be present. Then one line per goal.

In [15]:
tables = {path.stem: pd.read_csv(path) for path in sorted(SUMMARY.glob("*.csv"))}
named_sources = {part for table in tables.values() for column in table.columns if column.startswith("source")
                 for cell in table[column].dropna() for part in str(cell).split("; ") if part}


def present(path):
    """A named source exists: results/m4/... in RESULTS, anything else in the repo."""
    inside = path.startswith(NAMED_AS + "/")
    return (RESULTS / Path(path).relative_to(NAMED_AS) if inside else Path("..") / path).exists()


missing = sorted(path for path in named_sources if not present(path))
assert not missing, f"sources that do not exist: {missing}"
SHARES = {"arms": ["macro_f1", "who_recall_dr0", "top1", "top1_swapped", "top1_blanked"],
          "full_runs": ["top1", "top3", "macro_f1", "cal_who_lower_bound", "val_who_point", "val_false_alarm"],
          "gates": ["recall_calibration", "fpr_nonsnake_val", "recall_snake_val"],
          "gate_categories": [c for c in tables["gate_categories"].columns if c.endswith(("_val", "_test"))
                              and not c.startswith("photos")]}
for name, columns in SHARES.items():
    values = tables[name][columns]
    assert values.notna().all().all() and values.ge(0).all().all() and values.le(1).all().all(), name
numbered = goals[goals["goal"].ne("G3")]
assert numbered["mark"].notna().all(), "a goal has no mark"
assert numbered.loc[numbered["goal"].ne("G5"), "value"].notna().all(), "a goal has no number"
print(f"{len(named_sources)} source files named, all present; {sum(len(t) for t in tables.values())} rows in "
      f"{len(tables)} tables")
for row in goals.itertuples():
    verdict = "passed" if row.passed else "not passed"
    number = "" if pd.isna(row.value) else f"{row.value:.4g} ({row.rule} {row.mark:.4g}); "
    print(f"{row.goal}: {verdict}. {number}{row.detail} [{row.source}]")

35 source files named, all present; 171 rows in 16 tables


G1: passed. 0.05189 (at most 0.08396); top-1 0.6825 as is, 0.6306 swapped, 15977 photos; the mark is 0.5 of M3's gap [results/m4/full/A3/robustness_metrics.json]
G2: passed. 87 (at least 80); 87 of 158 measurable species [results/m4/full/A3/per_class_counts.json]
G3: not passed. top1 -0.081 (-0.100 to -0.064); macro_f1 -0.025 (-0.043 to -0.001); top1 lower, macro_f1 lower; 11931 photos; macro_f1 is per-species F1 over the photos of B1's species only. By M3's whole-slice definition (reported, not judged): +0.025 (+0.007 to +0.050), higher [results/m4/full/A3/b1_62.json]
G4: not passed. 0.2833 (at most 0.05); 170 of 600 test photos; validation snake recall 0.9828, mark at least 0.99 [results/m4/gate/non_snake_test.json]
G5: not passed. 2 (at most 1); 6 real test clips, 5 answered, 1 never answered (no answer, not within G5); changes per answered clip [2, 0, 0, 0, 0]; median first answer 0.50 s; final answer right on 1 of 4 species-scored clips [results/m4/video/A3/video_lock.csv]
Safety